# 05b — Comparação do híbrido original com a taxonomia parcial

Auditoria rápida, somente leitura, para decidir se o notebook 06 executado com o híbrido original pode ser interpretado com a taxonomia derivada pelo 05. A comparação usa todas as classes, alinha pelo `id` e reproduz separadamente a normalização de ranks do M01 e a do 06. **Igualdade de contagens não comprova igualdade do mapeamento.**

Por padrão, seleciona as execuções observadas `m01-a5e1773b35ab421a` e `c09-20261122T165714316366`. Para outras execuções, configure `M01_RUN_ID` e `M06_RUN_ID` antes de rodar. `DATA_ROOT`, `TAXONOMY_SOURCE_PATH` e `PARTIAL_TAXONOMY_PATH` podem substituir os caminhos; nenhum arquivo é alterado.

**Escopo:** verifica identidade Species, `model_label`, Genus, Order e os hashes registrados nos manifestos. Não demonstra independência genômica nem corrige a diferença de populações na correlação Genus → Order.


In [1]:
import hashlib
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "src" / "viral_cnn").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Execute a partir de um diretório do repositório cnn_metagenome")

DATA_ROOT = Path(os.environ.get("DATA_ROOT", str(PROJECT_ROOT.parent))).expanduser().resolve()
M01_RUN_ID = os.environ.get("M01_RUN_ID", "m01-a5e1773b35ab421a")
M06_RUN_ID = os.environ.get("M06_RUN_ID", "c09-20261122T165714316366")

source_path = Path(os.environ.get(
    "TAXONOMY_SOURCE_PATH",
    str(DATA_ROOT / "cnn_kraken/data_taxonomy_parquet/main_taxonomy_hybrid_ictv_ncbi.parquet"),
)).expanduser().resolve()
m01_manifest_path = PROJECT_ROOT / "results_reproduction/runs" / M01_RUN_ID / "short/05/manifest.json"
partial_path = Path(os.environ.get(
    "PARTIAL_TAXONOMY_PATH",
    str(m01_manifest_path.parent / "main_taxonomy_hybrid_PARTIAL.parquet"),
)).expanduser().resolve()
m06_manifest_path = (PROJECT_ROOT / "results_reproduction/runs" / M06_RUN_ID /
                     "06_ictv_latent_space_and_hierarchy/manifest.json")

for path in (source_path, partial_path, m01_manifest_path):
    if not path.is_file():
        raise FileNotFoundError(f"Entrada obrigatória ausente: {path}")
if source_path == partial_path:
    raise ValueError("Os dois caminhos apontam para o mesmo arquivo")

print("Híbrido original:", source_path)
print("Derivado do 05:", partial_path)
print("Manifesto 05:", m01_manifest_path)
print("Manifesto 06:", m06_manifest_path if m06_manifest_path.is_file() else "ausente")


Híbrido original: /home/mlima/workspace/cnn_kraken/data_taxonomy_parquet/main_taxonomy_hybrid_ictv_ncbi.parquet
Derivado do 05: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/m01-a5e1773b35ab421a/short/05/main_taxonomy_hybrid_PARTIAL.parquet
Manifesto 05: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/m01-a5e1773b35ab421a/short/05/manifest.json
Manifesto 06: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c09-20261122T165714316366/06_ictv_latent_space_and_hierarchy/manifest.json


In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def identity_for(manifest, path, field="inputs"):
    matches = [item for item in manifest.get(field, [])
               if Path(item.get("path", "")).expanduser().resolve() == path]
    if len(matches) != 1:
        return None
    return matches[0].get("sha256")


m01 = json.loads(m01_manifest_path.read_text(encoding="utf-8"))
m06 = json.loads(m06_manifest_path.read_text(encoding="utf-8")) if m06_manifest_path.is_file() else None
source_sha = sha256_file(source_path)
partial_sha = sha256_file(partial_path)

provenance = {
    "m01_complete_empirical": m01.get("status") == "COMPLETE" and m01.get("is_smoke_test") is False,
    "m01_partial_protocol": m01.get("protocol_id") == "m01_existing_partial_derive_v1",
    "m01_mapping_integrity": m01.get("metrics_summary", {}).get("mapping_integrity_ok") is True,
    "m01_source_sha_verified": identity_for(m01, source_path) == source_sha,
    "m01_partial_sha_verified": identity_for(m01, partial_path, field="outputs") == partial_sha,
    "m06_manifest_present": m06 is not None,
    "m06_empirical": m06 is not None and m06.get("is_smoke_test") is False,
    "m06_uses_this_source_sha": m06 is not None and identity_for(m06, source_path) == source_sha,
    "m06_uses_this_partial_sha": m06 is not None and identity_for(m06, partial_path) == partial_sha,
}
print("SHA256 original:", source_sha)
print("SHA256 derivado:", partial_sha)
display(pd.DataFrame([{"verificacao": k, "resultado": v} for k, v in provenance.items()]))


SHA256 original: a062cafffa378e4a54c24c9280503f1a39ff0d57198190f7cffd48d07be6dd4c
SHA256 derivado: d8ab34fd9c1e43b2dc0be7fe23589f6bacb8b638dfc968b2aa5855363608068a


,verificacao,resultado
0,m01_complete_empirical,True
1,m01_partial_protocol,True
2,m01_mapping_integrity,True
3,m01_source_sha_verified,True
4,m01_partial_sha_verified,True
5,m06_manifest_present,True
6,m06_empirical,True
7,m06_uses_this_source_sha,True
8,m06_uses_this_partial_sha,False


In [3]:
def indexed_taxonomy(path):
    frame = pd.read_parquet(path)
    required = {"id", "species", "Genus", "Order"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{path.name}: colunas obrigatórias ausentes: {sorted(missing)}")
    ids = pd.to_numeric(frame["id"], errors="coerce")
    if (ids.isna().any() or (ids < 1).any() or (ids % 1 != 0).any()
            or ids.duplicated().any()):
        raise ValueError(f"{path.name}: IDs inválidos ou repetidos")
    frame = frame.copy()
    frame["id"] = ids.astype("int64")
    if "model_label" in frame.columns:
        labels = pd.to_numeric(frame["model_label"], errors="coerce")
        if labels.isna().any() or not labels.eq(ids - 1).all():
            raise ValueError(f"{path.name}: model_label não corresponde a id - 1")
        frame["model_label"] = labels.astype("int64")
    else:
        frame["model_label"] = frame["id"] - 1
    return frame.set_index("id").sort_index()


def normalize_rank(series, *, as_m01):
    values = series.astype("string").str.strip()
    placeholders = ("Unassigned", "Unclassified") if as_m01 else ("Unassigned",)
    missing = (values.isna() | values.eq("").fillna(False)
               | values.str.startswith(placeholders, na=False))
    return values.mask(missing, pd.NA)


def mismatch_mask(left, right):
    return ~(left.eq(right).fillna(False) | (left.isna() & right.isna()))


original = indexed_taxonomy(source_path)
partial = indexed_taxonomy(partial_path)
all_ids = original.index.union(partial.index)
left = original.reindex(all_ids)
right = partial.reindex(all_ids)

masks = {
    "presenca_id": pd.Series(~all_ids.isin(original.index.intersection(partial.index)), index=all_ids),
    "species": mismatch_mask(left["species"].astype("string"), right["species"].astype("string")),
    "model_label": mismatch_mask(left["model_label"].astype("Int64"), right["model_label"].astype("Int64")),
}
for rank in ("Genus", "Order"):
    masks[f"{rank}_como_M01"] = mismatch_mask(
        normalize_rank(left[rank], as_m01=True), normalize_rank(right[rank], as_m01=True))
    masks[f"{rank}_como_06"] = mismatch_mask(
        normalize_rank(left[rank], as_m01=False), normalize_rank(right[rank], as_m01=False))

counts = {key: int(mask.sum()) for key, mask in masks.items()}
display(pd.DataFrame([{"comparacao": k, "ids_diferentes": v} for k, v in counts.items()]))
print("Linhas no original:", len(original), "| no derivado:", len(partial))
for rank in ("Genus", "Order"):
    for name, frame in (("original usado pelo 06", original), ("derivado do 05", partial)):
        mapped = int(normalize_rank(frame[rank], as_m01=(name == "derivado do 05")).notna().sum())
        print(f"{name} — {rank}: {mapped} mapeadas; {len(frame)-mapped} excluídas")

any_mismatch = pd.concat(masks.values(), axis=1).any(axis=1)
if any_mismatch.any():
    examples = pd.DataFrame({"id": all_ids[any_mismatch],
                             "species_original": left.loc[any_mismatch, "species"].array,
                             "species_derivada": right.loc[any_mismatch, "species"].array,
                             "Genus_original": left.loc[any_mismatch, "Genus"].array,
                             "Genus_derivado": right.loc[any_mismatch, "Genus"].array,
                             "Order_original": left.loc[any_mismatch, "Order"].array,
                             "Order_derivado": right.loc[any_mismatch, "Order"].array})
    print(f"Exibindo 20 dos {int(any_mismatch.sum())} IDs com diferença:")
    display(examples.head(20))


,comparacao,ids_diferentes
0,presenca_id,0
1,species,0
2,model_label,0
3,Genus_como_M01,0
4,Genus_como_06,0
5,Order_como_M01,0
6,Order_como_06,0


Linhas no original: 13683 | no derivado: 13683
original usado pelo 06 — Genus: 7746 mapeadas; 5937 excluídas
derivado do 05 — Genus: 7746 mapeadas; 5937 excluídas
original usado pelo 06 — Order: 4583 mapeadas; 9100 excluídas
derivado do 05 — Order: 4583 mapeadas; 9100 excluídas


In [4]:
mapping_equal = all(value == 0 for value in counts.values())
m01_verified = all(provenance[k] for k in (
    "m01_complete_empirical", "m01_partial_protocol", "m01_mapping_integrity",
    "m01_source_sha_verified", "m01_partial_sha_verified"))
m06_source_verified = provenance["m06_empirical"] and provenance["m06_uses_this_source_sha"]

if not mapping_equal:
    verdict = "DIFERENTES: não trate a hierarquia do 06 como equivalente à derivada pelo 05"
elif not m01_verified or not m06_source_verified:
    verdict = "MAPEAMENTO IGUAL, PROVENIÊNCIA INCOMPLETA: confira os manifestos"
else:
    verdict = "EQUIVALENTES PARA OS RANKS DO 06: mantenha explícito que o 06 leu o original auditado pelo 05"

print("VEREDITO:", verdict)
print("A igualdade dos Parquets não resolve a diferença de populações usada na correlação Genus → Order.")


VEREDITO: EQUIVALENTES PARA OS RANKS DO 06: mantenha explícito que o 06 leu o original auditado pelo 05
A igualdade dos Parquets não resolve a diferença de populações usada na correlação Genus → Order.
